# 03. Cortex Search ― PDF レポートを検索できるようにする

02 の Cortex Analyst は「テーブルの数字」に答えます。一方、**「推奨カバレッジは？」「停滞商談にはどう対応すべき？」** のような答えは PDF のレポートにあります。
**Cortex Search** は、文書を意味で検索できるようにするサービスです。

```
PDF（DOCS_STAGE。中身は GitHub の data/pdfs/ で見られます）
  ↓ ① AI_PARSE_DOCUMENT   … PDF からテキストを抽出（表は Markdown に）
  ↓ ② SPLIT_TEXT_RECURSIVE_CHARACTER … 検索しやすい長さ（チャンク）に分割
  ↓ ③ CREATE CORTEX SEARCH SERVICE  … ベクトル＋キーワードのハイブリッド検索インデックス
Cortex Search Service `SALES_DOCS_SEARCH` → 04 で Agent のツールにする
```

**前提**：`02_cortex_analyst.ipynb` を実行済みであること

## Step 0. 実行環境をそろえる

使うロール・ウェアハウス・スキーマを指定します。Notebook はそれぞれ独立して動くため、各 Notebook の最初に実行します。

In [ ]:
%%sql -r setup
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE CORTEX_QUICK_HOL_WH;
USE SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA;
SELECT CURRENT_ROLE() AS "ロール", CURRENT_WAREHOUSE() AS "ウェアハウス", CURRENT_SCHEMA() AS "スキーマ"

## Step 1. PDF からテキストを取り出す

`AI_PARSE_DOCUMENT` の `LAYOUT` モードは、見出しや**表の構造を保ったまま** Markdown のテキストにします。

In [ ]:
%%sql -r parse_docs
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED AS
SELECT RELATIVE_PATH AS FILE_NAME,
       AI_PARSE_DOCUMENT(TO_FILE('@CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_STAGE', RELATIVE_PATH), {'mode': 'LAYOUT'}):content::STRING AS CONTENT
FROM DIRECTORY(@CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_STAGE)

In [ ]:
%%sql -r show_parsed
SELECT FILE_NAME, LENGTH(CONTENT) AS CHARS, LEFT(CONTENT, 500) AS HEAD FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED

## Step 2. チャンクに分割する

PDF 1 本を丸ごと 1 件として登録せず、短い「チャンク」に分けてから検索サービスに入れます。理由は 3 つです。

| 理由 | 内容 |
|---|---|
| ベクトル化できる長さに上限がある | 一度に読める長さは埋め込みモデルによって違い、今回使う `snowflake-arctic-embed-l-v2.0` は **512 トークン**です（`-8k` 版は 8,192、`voyage-multilingual-2` は 32,000）。上限を超えた部分は**ベクトル検索では使われません**（キーワード検索では全文が使われます）。長く読めるモデルでも、Snowflake は 1 チャンク 512 トークン以下を推奨しています |
| 検索の精度が上がる | 1 件の中に話題がたくさん混ざると、質問との「意味の近さ」がぼやけます。話題ごとに分けると、関係する段落だけが上位に来ます |
| Agent に渡す量が適切になる | Agent は検索結果を読んで回答を作ります。PDF 全体ではなく関係する段落だけを渡すほうが、回答が正確で速くなります |

ここでは約 800 文字ずつ（前後 100 文字は重ねて、文の途中で切れても前後の文脈が残るように）分割します。
`'markdown'` を指定すると、見出しや段落の切れ目を優先して分割します。

In [ ]:
%%sql -r chunk_docs
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS AS
SELECT d.FILE_NAME || '#' || c.INDEX AS CHUNK_ID,
       d.FILE_NAME,
       c.INDEX        AS CHUNK_NO,
       c.VALUE::STRING AS CHUNK
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED d,
     LATERAL FLATTEN(SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(d.CONTENT, 'markdown', 800, 100)) c

In [ ]:
%%sql -r chunk_counts
SELECT FILE_NAME, COUNT(*) AS CHUNKS, ROUND(AVG(LENGTH(CHUNK))) AS AVG_CHARS
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS GROUP BY FILE_NAME

## Step 3. Cortex Search Service を作る

Step 2 の `DOCS_CHUNKS` を検索サービスにします。決めることは次の 5 つで、GUI の各ステップと SQL の各句は 1 対 1 で対応しています。

| GUI のステップ | SQL | 今回の値 | 何のために決めるか |
|---|---|---|---|
| データを選択 | `AS SELECT ... FROM` | `DOCS_CHUNKS` テーブル | 検索対象にする文書（1 行 = 1 チャンク）を決める |
| 検索列を選択 | `ON` | `CHUNK` | **質問と照らし合わせる本文**。この列がベクトル化され、意味とキーワードの両方で検索されます |
| 属性を選択 | `ATTRIBUTES` | `FILE_NAME` | **絞り込みに使う列**。「Sales Performance Analysis だけから探す」のように、検索時にフィルタをかけられます |
| 列を選択 | `AS SELECT` の列 | `CHUNK_ID` / `CHUNK` / `FILE_NAME` / `CHUNK_NO` | **検索結果として返す列**。04 の Agent は `CHUNK_ID` を出典の ID、`FILE_NAME` を出典名として使うので、必ず含めます |
| インデックス作成を構成 | `WAREHOUSE` / `TARGET_LAG` / `EMBEDDING_MODEL` | `CORTEX_QUICK_HOL_WH` / `1 day` / `snowflake-arctic-embed-l-v2.0` | 索引を作るウェアハウス、元データの変更に追従する間隔、ベクトル化に使うモデル |

> 💡 **埋め込みモデルは多言語のものを選びます。** 既定のモデルは英語向けで、このハンズオンで試したところ、日本語の質問に対して関係の薄い表が上位に来ました。`snowflake-arctic-embed-l-v2.0` なら、英語の PDF を日本語の質問で検索できます。
>
> 作成には 1〜2 分かかります。

### 方法 A：GUI（AI と ML → Cortex Search）で作る

1. Snowsight 左メニュー **「AI と ML」→「Cortex Search」** を開き、**「作成」**
2. **新しいサービス**：ロール `ACCOUNTADMIN`、ウェアハウス `CORTEX_QUICK_HOL_WH`、データベース / スキーマ `CORTEX_QUICK_HOL_DB` / `CORTEX_QUICK_HOL_SCHEMA`、サービス名 **`SALES_DOCS_SEARCH`**（**大文字のまま**）
3. **データを選択**：テーブル **`DOCS_CHUNKS`**
4. **検索列を選択**：**`CHUNK`**
5. **属性を選択**：**`FILE_NAME`**
6. **列を選択**：`CHUNK_ID` / `CHUNK` / `FILE_NAME` / `CHUNK_NO` の 4 つ
7. **インデックス作成を構成**：ターゲットラグ **1 day**。埋め込みモデルを選べる場合は **`snowflake-arctic-embed-l-v2.0`** → **作成**
8. 作成されたら同じ画面の **Playground** で「推奨されるパイプラインカバレッジの目安は？」と検索し、1.6 倍が書かれた段落が上位に来ることを確かめる

> ⚠️ 埋め込みモデルを選べなかった場合は英語向けのモデルで作られるため、**方法 B のセルを実行して作り直してください**（同じ名前で上書きされます）。
>
> ⚠️ GUI ではサービス名がダブルクォート付きで作られます。小文字が混じると SQL や Agent から `SALES_DOCS_SEARCH` で参照できなくなるため、**必ず大文字で入力**してください。

### 方法 B：SQL で作る（`CREATE CORTEX SEARCH SERVICE`）

In [ ]:
%%sql -r create_search_service
CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH
  ON CHUNK
  ATTRIBUTES FILE_NAME
  WAREHOUSE = CORTEX_QUICK_HOL_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS SELECT CHUNK_ID, CHUNK, FILE_NAME, CHUNK_NO FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS

GUI・SQL どちらで作った場合も、次のセルで権限を付けます。

In [ ]:
%%sql -r grant_search
GRANT USAGE ON CORTEX SEARCH SERVICE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH TO ROLE CORTEX_QUICK_HOL_ROLE

## Step 4. 検索してみる

`SEARCH_PREVIEW` で、質問に近いチャンクを上位から取り出します。英語の PDF に**日本語で**質問しています。
質問を変えて試す場合は、SQL の `query` の部分を書き換えて実行してください。

In [ ]:
%%sql -r search_coverage
SELECT r.index + 1 AS "順位",
       r.value:FILE_NAME::STRING AS "ファイル",
       LEFT(REGEXP_REPLACE(r.value:CHUNK::STRING, '\\s+', ' '), 300) AS "本文（先頭 300 文字）"
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH',
  '{"query": "推奨されるパイプラインカバレッジの目安は？", "columns": ["CHUNK", "FILE_NAME"], "limit": 3}'
)):results)) r

In [ ]:
%%sql -r search_stalled
SELECT r.index + 1 AS "順位",
       r.value:FILE_NAME::STRING AS "ファイル",
       LEFT(REGEXP_REPLACE(r.value:CHUNK::STRING, '\\s+', ' '), 300) AS "本文（先頭 300 文字）"
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH',
  '{"query": "停滞している商談にはどう対応すべき？", "columns": ["CHUNK", "FILE_NAME"], "limit": 3}'
)):results)) r

In [ ]:
%%sql -r search_churn
SELECT r.index + 1 AS "順位",
       r.value:FILE_NAME::STRING AS "ファイル",
       LEFT(REGEXP_REPLACE(r.value:CHUNK::STRING, '\\s+', ' '), 300) AS "本文（先頭 300 文字）"
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH',
  '{"query": "解約リスクが高い顧客に共通する特徴は？", "columns": ["CHUNK", "FILE_NAME"], "limit": 3}'
)):results)) r

> ✅ それぞれの質問に、関係する段落（推奨 1.6 倍、役員スポンサーの関与、技術的なエスカレーションなど）が上位に返れば成功です。

✅ 03 はここまでです。次は `04_cortex_agent.ipynb` で、Analyst と Search を 1 つの Agent にまとめます。